## Config & Data (Roboflow)

In [ ]:
!pip install ultralytics roboflow -q

In [ ]:
# Pull dataset directly from Roboflow, which lets it enforce a real
# train/valid/test split (no more train==val leakage).
from roboflow import Roboflow

rf = Roboflow(api_key="YOUR_API_KEY")
project = rf.workspace("YOUR_WORKSPACE").project("YOUR_PROJECT")
version = project.version(1)  # set your version number
dataset = version.download("yolov11")  # yolo-format export works for yolo26 too

dataset_yaml_path = f"{dataset.location}/data.yaml"
print(dataset_yaml_path)

In [ ]:
# Sanity check: confirm both classes actually have labeled instances.
# This catches empty-class problems (like the accidental smoke-label
# deletion in the old notebook) before you burn GPU hours on it.
import yaml, glob
from collections import Counter

with open(dataset_yaml_path) as f:
    cfg = yaml.safe_load(f)
print("Classes:", cfg['names'])

counts = Counter()
for split in ['train', 'val', 'valid', 'test']:
    label_dir = f"{dataset.location}/{split}/labels"
    for txt in glob.glob(f"{label_dir}/*.txt"):
        with open(txt) as f:
            for line in f:
                if line.strip():
                    counts[(split, line.split()[0])] += 1
print("Per-split class counts:", dict(counts))
assert len(counts) > 0, "No labels found - check dataset export/paths"

## Train

In [ ]:
from google.colab import drive
from ultralytics import YOLO
from pathlib import Path

drive.mount('/content/drive')

ROOT = Path.cwd()
OUTPUT_DIR = ROOT / "output"

best_model = "/content/drive/MyDrive/Models/fireseg_v2.pt"
model = YOLO(best_model)

In [ ]:
# Config
model_variant = "yolo26n-seg"
img_size = 640
epochs = 500
batch_size = 64
seed = 42  # fixed seed -> reproducible metrics across runs

run_name = f"{model_variant}_{img_size}_e{epochs}_b{batch_size}"

results = model.train(
    data=dataset_yaml_path,
    epochs=epochs,
    imgsz=img_size,
    batch=batch_size,
    project=str(OUTPUT_DIR),
    name=run_name,
    val=True,
    seed=seed,
    patience=50,        # early stopping: halt if no val improvement for 50 epochs
    plots=True,          # forces PR curve / confusion matrix / F1 curve generation
    save_period=50        # periodic checkpoints, useful if Colab disconnects
)

## Metrics

Ultralytics writes `results.csv`, `confusion_matrix.png`, `PR_curve.png`, `F1_curve.png`, and per-class metrics into the run folder automatically. This cell just surfaces the numbers that matter most for a fire/smoke detector: **per-class mAP50, mAP50-95, precision, recall**, plus a note on why each matters.

In [ ]:
# results.box.maps -> mAP50-95 per class (segmentation: use results.seg.*)
metrics = model.val(data=dataset_yaml_path, split="test")  # run on held-out test split, not val

print("== Segmentation metrics (mask) ==")
print("mAP50-95:", metrics.seg.map)
print("mAP50   :", metrics.seg.map50)
print("Precision (per class):", metrics.seg.p)
print("Recall    (per class):", metrics.seg.r)

for i, name in enumerate(cfg['names']):
    print(f"\nClass '{name}':")
    print(f"  mAP50-95: {metrics.seg.maps[i]:.4f}")

print("\nNote: for fire/smoke safety applications, RECALL usually matters")
print("more than precision -- a missed fire (false negative) is far more")
print("costly than a false alarm. Watch per-class recall closely, and")
print("check the confusion matrix in the run folder for which class is")
print("being missed.")

## Save

In [ ]:
!zip -rq /content/output.zip /content/output
!cp /content/output.zip "/content/drive/MyDrive/Datasets/"